# 17.7 先只量化權重可以嗎？

# 第 17 章：量化，用更少位元表示模型

前置：第2章Linear與數值。像把連續刻度尺改成較粗的格子：先看誤差，再看真正的儲存bytes，最後才談硬體加速。quantization.py的參考Linear會反量化，不會假裝是低位元kernel。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：換較粗刻度尺：省格子但有誤差**

縮小儲存與硬體加速分開驗證。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/quantization.svg")))

**先想一想：**權重buffer是uint8，乘法就自動int4了嗎？

weight-only先壓縮weight，activation仍浮點。參考forward先反量化再Linear，用它檢查誤差，但不預設執行更快。

**把直覺寫成數學：**y=x·dequantize(qW)ᵀ+b；運算仍是浮點。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.quantization import QuantizedLinear

layer = nn.Linear(32, 16)
compressed = QuantizedLinear(layer, 4)
x = torch.randn(2, 32)
print(
    "stored",
    compressed.values.dtype,
    "output",
    compressed(x).dtype,
    "MAE",
    (layer(x) - compressed(x)).abs().mean().item(),
)

**如何讀結果：**明確分開存儲dtype與compute dtype；專門kernel放17.10的硬體延伸。

**只改一件事：**只改爲8bit，比較誤差與buffer bytes。
